---
format: html
---

# Exercise bank {.unnumbered}

<div id="question-bank" style="display: none !important;">

{{< include ../exercises/exercises.qmd >}}

</div>

<div id="exercise-app-root" class="my-4"></div>

```{=html}
<script src="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/highlight.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/languages/julia.min.js"></script>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/styles/github.min.css">

<script>
document.addEventListener("DOMContentLoaded", function () {
  const SERVER_SYNC_URL = "https://jack.thomaslabs.co.uk/Math5485-gh/backup.php";

  // Solutions are released one week after the lecture, at 9am (Minneapolis time).
  // Exercises marked release="now" (e.g. quiz questions) have their solutions available immediately.
  // Lecture dates (see the course schedule in the syllabus):
  const SOLUTION_DELAY_DAYS = 7;
  const LECTURE_DATES = {
     1: '2026-09-09',  2: '2026-09-14',  3: '2026-09-16',  4: '2026-09-21',
     5: '2026-09-23',  6: '2026-09-28',  7: '2026-10-05',  8: '2026-10-07',
     9: '2026-10-12', 10: '2026-10-14', 11: '2026-10-19', 12: '2026-10-21',
    13: '2026-11-09', 14: '2026-11-11', 15: '2026-11-23', 16: '2026-11-30',
    17: '2026-12-02', 18: '2026-12-07', 19: '2026-12-09', 20: '2026-12-14'
  };

  const root = document.getElementById('exercise-app-root');
  const sourcePool = document.getElementById('question-bank');

  if (!root || !sourcePool) return;

  function getFormattedStudentId() {
    let id = localStorage.getItem('studentID');
    if (!id || !/^\d{3}-\d{3}-\d{3}$/.test(id)) {
      const pad = (num) => String(num).padStart(3, '0');
      const p1 = pad(Math.floor(Math.random() * 1000));
      const p2 = pad(Math.floor(Math.random() * 1000));
      const p3 = pad(Math.floor(Math.random() * 1000));
      id = `${p1}-${p2}-${p3}`;
      localStorage.setItem('studentID', id);
    }
    return id;
  }

  let studentID = getFormattedStudentId();
  const allExercises = Array.from(sourcePool.querySelectorAll('.exercise-box'));

  if (allExercises.length === 0) {
    root.innerHTML = '<div class="alert alert-warning">No exercises found.</div>';
    return;
  }

  function cleanTitle(rawText) {
    return rawText.replace(/^(\d+(\.\d+)*\.?\s*|chapter\s+\d+:?\s*|section\s+\d+:?\s*)/i, '').trim();
  }

  let currentLecture = "Exercises";

  allExercises.forEach((el, index) => {
    let prev = el.previousElementSibling;
    while (prev) {
      if (/^H[1-6]$/i.test(prev.tagName) || prev.classList.contains('lecture-title')) {
        currentLecture = cleanTitle(prev.textContent);
        break;
      }
      prev = prev.previousElementSibling;
    }

    if (!el.dataset.lecture) el.dataset.lecture = currentLecture;

    const titleHeader = el.querySelector('h1, h2, h3, h4, .exercise-title, strong');
    const displayLabel = titleHeader ? titleHeader.textContent.trim() : `Ex ${index + 1}`;
    el.dataset.title = displayLabel;

    if (!el.id) {
      const textContent = el.textContent.trim();
      let hash = 0;
      for (let i = 0; i < textContent.length; i++) {
        hash = ((hash << 5) - hash) + textContent.charCodeAt(i);
        hash |= 0;
      }
      el.id = 'exr_hash_' + Math.abs(hash);
    }
  });

  // Inject UI structure
  root.innerHTML = `
    <style>
      .exr-nav-btn {
        display: inline-flex; align-items: center; gap: 6px;
        padding: 7px 16px; border-radius: 999px; font-size: 0.875rem;
        font-weight: 500; border: 1.5px solid; cursor: pointer;
        transition: background 0.15s, box-shadow 0.15s, transform 0.1s;
        box-shadow: 0 1px 3px rgba(0,0,0,0.08);
        background: #fff;
      }
      .exr-nav-btn:hover { box-shadow: 0 3px 8px rgba(0,0,0,0.13); transform: translateY(-1px); }
      .exr-nav-btn:active { transform: translateY(0); box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
      .exr-nav-btn.prev  { border-color: #6c757d; color: #6c757d; }
      .exr-nav-btn.prev:hover  { background: #6c757d; color: #fff; }
      .exr-nav-btn.next  { border-color: #0d6efd; color: #0d6efd; }
      .exr-nav-btn.next:hover  { background: #0d6efd; color: #fff; }
      .exr-nav-btn.rand  { border-color: #0d6efd; background: #0d6efd; color: #fff; }
      .exr-nav-btn.rand:hover  { background: #0b5ed7; border-color: #0b5ed7; }
      .exr-nav-btn svg { flex-shrink: 0; }
      .exr-nav-btn:disabled { opacity: 0.4; cursor: not-allowed; box-shadow: none; transform: none; }
      .exr-nav-btn:disabled:hover { background: #fff; color: inherit; }
      #exr-progress { display: flex; align-items: center; gap: 0.6rem; margin: 0.35rem 0 0.6rem; }
      #exr-progress[hidden] { display: none; }
      #exr-progress-label { font-size: 0.8rem; color: #6c757d; white-space: nowrap; }
      #exr-progress-bar { flex: 1; display: flex; gap: 3px; height: 8px; }
      .exr-seg { flex: 1; min-width: 6px; height: 100%; padding: 0; border: none; border-radius: 4px;
                 background: #e9ecef; cursor: pointer; transition: background 0.15s; }
      .exr-seg.passed  { background: #9ec5fe; }
      .exr-seg.up      { background: #75b798; }                     /* completed */
      .exr-seg.down    { background: #ffc107; }                     /* needs review */
      .exr-seg.current { box-shadow: 0 0 0 2px #fff, 0 0 0 4px #0d6efd; }
      .exr-seg.current:not(.up):not(.down) { background: #0d6efd; }
      .exr-seg:not(.current):not(:disabled):hover { filter: brightness(0.85); }
      .exr-seg:disabled { opacity: 0.35; cursor: default; }
      .exr-nav-btn.hint { border-color: #f0ad4e; color: #b07800; }
      .exr-nav-btn.hint:hover { background: #f0ad4e; color: #fff; }
      .exr-nav-btn.hint.hint-on { background: #f0ad4e; color: #fff; border-color: #f0ad4e; }
      #hint-container { border: none; background: transparent; border-radius: 0; padding: 0; margin-top: 0.4rem; }
      #solution-container { border: none; background: #f0faf4; border-radius: 8px; padding: 1rem 1.25rem; margin-top: 0.75rem; }
      /* Hint / Solution buttons at the bottom (centre) of the question box */
      .exr-box-actions { display: flex; justify-content: center; gap: 0.5rem; flex-wrap: wrap; margin-top: 0.4rem; }
      .exr-box-actions[hidden] { display: none; }
      .exr-nav-btn.next.active { background: #0d6efd; color: #fff; }
      /* Completed / Needs review buttons (between the lecture buttons) */
      .exr-status { display: flex; gap: 0.5rem; justify-content: center; flex-wrap: wrap; }
      .exr-nav-btn.status-up   { border-color: #198754; color: #198754; }
      .exr-nav-btn.status-up:hover, .exr-nav-btn.status-up.active { background: #198754; color: #fff; }
      .exr-nav-btn.status-down { border-color: #e0a800; color: #9a6b00; }
      .exr-nav-btn.status-down:hover, .exr-nav-btn.status-down.active { background: #ffc107; border-color: #ffc107; color: #212529; }
      /* Difficulty stars in the top-right corner of the question box */
      .exr-difficulty { position: absolute; top: 0.55rem; right: 0.75rem; display: flex; align-items: center; gap: 0.35rem;
                        font-size: 0.75rem; color: #6c757d; }
      .exr-stars { display: flex; }
      .exr-stars button { background: none; border: none; padding: 0 1px; font-size: 1.15rem; line-height: 1;
                          color: #f0ad4e; cursor: pointer; transition: transform 0.1s; }
      .exr-stars button:hover { transform: scale(1.2); }
      #random-exercise-display .exercise-box > p:first-of-type { padding-right: 10.5rem; }
      @media (max-width: 575.98px) {
        /* phones: Completed / Needs review on their own row above the lecture buttons */
        #section-nav .exr-status { order: -1; width: 100%; }
        .exr-difficulty-label { display: none; }
        #random-exercise-display .exercise-box > p:first-of-type { padding-right: 6.5rem; }
      }
      .bank-help { font-size: 0.9rem; margin: 0.25rem 0 0.5rem; }
      .bank-help summary { cursor: pointer; color: #0d6efd; width: fit-content; }
      .bank-help ul { margin: 0.5rem 0 0; padding-left: 1.25rem; }
      .bank-help li { margin-bottom: 0.25rem; }
      .bank-help kbd, .kbd-hint kbd { font-size: 0.75em; padding: 1px 5px; }
      .cnt-btn { background: none; border: none; padding: 0; color: inherit; font: inherit; cursor: pointer; }
      .cnt-btn:hover { text-decoration: underline; }
      .cnt-btn.active { text-decoration: underline; text-decoration-thickness: 2px; color: #212529; }
      .id-panel { background: #f8f9fa; border: 1px solid #dee2e6; border-radius: 8px; padding: 0.75rem 1rem; margin-top: 0.75rem; }
      .id-panel[hidden] { display: none; }
      .id-panel code { font-size: 0.95rem; color: #0d6efd; background: #fff; border: 1px solid #dee2e6; border-radius: 4px; padding: 1px 6px; }
    </style>
    <details class="bank-help">
      <summary>How this works</summary>
      <ul>
        <li><b>Practise:</b> move between questions with <i>Previous</i> / <i>Next</i>, <i>Shuffle</i>, the progress bar, or <i>Go to</i>. The page remembers where you were, and each question has its own link.</li>
        <li><b>Track your progress:</b> mark each question <i>✓ Completed</i> or <i>⚠️ Needs review</i> and rate its difficulty. Use <i>Show</i> (or click the counters) to practise only unrated, needs-review or completed questions. In the progress bar, green = completed and amber = needs review.</li>
        <li><b>Hints and solutions:</b> hints are always available; solutions unlock one week after the lecture (quiz solutions straight away).</li>
        <li><b>Other devices:</b> your progress is saved in this browser. Use <i>🔑 ID</i> to load it on another device or browser.</li>
        <li class="kbd-hint"><b>Keyboard:</b> <kbd>←</kbd> / <kbd>→</kbd> previous / next question, <kbd>Shift</kbd> + <kbd>←</kbd> / <kbd>→</kbd> previous / next lecture, <kbd>S</kbd> shuffle, <kbd>H</kbd> hint.</li>
        <li><b>Privacy:</b> to help improve the course, your ratings, time spent per question and use of hints/solutions are sent to the course server under a random ID (like 123-456-789), not your name or student number.</li>
      </ul>
    </details>
    <div class="card p-3 my-3">
      <div class="d-flex align-items-center justify-content-between flex-wrap gap-3">
        <label class="d-flex align-items-center gap-2 small text-muted mb-0">Go to
          <select id="section-select" class="form-select form-select-sm" style="width: auto;"></select>
        </label>
        <label class="d-flex align-items-center gap-2 small text-muted mb-0">Show
          <select id="filter-select" class="form-select form-select-sm" style="width: auto;">
            <option value="all">All questions</option>
            <option value="unrated">❓ Unrated only</option>
            <option value="down">⚠️ Needs review only</option>
            <option value="up">✓ Completed only</option>
          </select>
        </label>
      </div>

      <div class="d-flex justify-content-between align-items-center gap-2 mt-3 pt-2 border-top small text-muted flex-wrap">
        <div>
          <button class="cnt-btn" data-filter="all" title="Show all questions">Total: <strong id="cnt-all">0</strong></button> |
          <button class="cnt-btn" data-filter="unrated" title="Show unrated questions only">Unrated: <strong id="cnt-unrated" class="text-secondary">0</strong></button> |
          <button class="cnt-btn" data-filter="up" title="Show completed questions only">Completed: <strong id="cnt-up" class="text-success">0</strong></button> |
          <button class="cnt-btn" data-filter="down" title="Show needs-review questions only">Needs review: <strong id="cnt-down" class="text-warning">0</strong></button>
        </div>
        <div class="d-flex align-items-center gap-2">
          <span id="sync-status" class="badge bg-light text-muted border" title="Your progress is saved in this browser and backed up anonymously (see How this works)">Synced</span>
          <span id="streak-display" title="Day streak" style="font-size:.82rem;color:#6c757d;"></span>
          <button id="manage-id-btn" class="btn btn-sm btn-outline-secondary py-0" title="Use your progress on another device" aria-expanded="false" aria-controls="id-panel">🔑 ID</button>
        </div>
      </div>

      <!-- ID panel: copy this device's ID, or load progress from another device's ID -->
      <div id="id-panel" class="id-panel small" hidden>
        <div class="d-flex flex-wrap align-items-center gap-2">
          <span>Your ID:</span> <code id="id-current"></code>
          <button id="id-copy" type="button" class="btn btn-sm btn-outline-secondary py-0">Copy</button>
        </div>
        <div class="text-muted mt-2">Enter this ID on another device (or browser) to see the same progress there. To load progress from another device, enter that device's ID:</div>
        <div class="d-flex flex-wrap align-items-center gap-2 mt-1">
          <input id="id-input" class="form-control form-control-sm" style="width: 9.5em" placeholder="123-456-789" maxlength="11" autocomplete="off">
          <button id="id-load" type="button" class="btn btn-sm btn-primary py-0">Load progress</button>
        </div>
        <div id="id-msg" class="mt-2" role="status"></div>
      </div>
    </div>

    <!-- Navigate by question: previous (left), shuffle (middle), next (right) -->
    <div id="question-nav" class="d-flex justify-content-between align-items-center gap-2 mb-2 flex-wrap">
          <button id="prev-exr-btn" class="exr-nav-btn prev" title="Previous question">
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M11.354 1.646a.5.5 0 0 1 0 .708L5.707 8l5.647 5.646a.5.5 0 0 1-.708.708l-6-6a.5.5 0 0 1 0-.708l6-6a.5.5 0 0 1 .708 0z"/></svg>
            Previous
          </button>
          <button id="random-exr-btn" class="exr-nav-btn rand" title="Random question">
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path fill-rule="evenodd" d="M0 3.5A.5.5 0 0 1 .5 3H1c2.202 0 3.827 1.24 4.874 2.418.49.552.865 1.102 1.126 1.532.26-.43.636-.98 1.126-1.532C9.173 4.24 10.798 3 13 3v1c-1.798 0-3.173 1.01-4.126 2.082A9.6 9.6 0 0 0 7.556 8a9.6 9.6 0 0 0 1.317 1.918C9.828 10.99 11.204 12 13 12v1c-2.202 0-3.827-1.24-4.874-2.418A10.6 10.6 0 0 1 7 9.05c-.26.43-.636.98-1.126 1.532C4.827 11.76 3.202 13 1 13H.5a.5.5 0 0 1 0-1H1c1.798 0 3.173-1.01 4.126-2.082A9.6 9.6 0 0 0 6.443 8a9.6 9.6 0 0 0-1.317-1.918C4.172 5.01 2.796 4 1 4H.5a.5.5 0 0 1-.5-.5z"/><path d="M13 5.466V1.534a.25.25 0 0 1 .41-.192l2.36 1.966c.12.1.12.284 0 .384l-2.36 1.966a.25.25 0 0 1-.41-.192zm0 9v-3.932a.25.25 0 0 1 .41-.192l2.36 1.966c.12.1.12.284 0 .384l-2.36 1.966a.25.25 0 0 1-.41-.192z"/></svg>
            Shuffle
          </button>
          <button id="next-exr-btn" class="exr-nav-btn next" title="Next question">
            Next
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M4.646 1.646a.5.5 0 0 1 .708 0l6 6a.5.5 0 0 1 0 .708l-6 6a.5.5 0 0 1-.708-.708L10.293 8 4.646 2.354a.5.5 0 0 1 0-.708z"/></svg>
          </button>
    </div>

    <!-- Progress through the current lecture/quiz: one segment per question -->
    <div id="exr-progress" hidden>
      <span id="exr-progress-label"></span>
      <div id="exr-progress-bar" role="progressbar" aria-valuemin="1"></div>
    </div>

    <!-- Container for active exercise -->
    <div id="random-exercise-display"></div>

    <!-- Hints & Solutions Control Bar -->
    <div id="solution-controls" class="my-3" style="display:none;">
      <!-- Hint / Solution buttons: moved to the bottom of the question box; the hint and
           solution themselves are shown below the box -->
      <div id="box-actions" class="exr-box-actions">
        <button id="btn-show-hint" class="exr-nav-btn hint" title="Toggle hint" style="display:none;">
          <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M2 6a6 6 0 1 1 10.174 4.31c-.203.196-.359.4-.453.619l-.762 1.769A.5.5 0 0 1 10.5 13h-5a.5.5 0 0 1-.46-.302l-.761-1.77a2 2 0 0 0-.453-.618A5.98 5.98 0 0 1 2 6zm3 8.5a.5.5 0 0 1 .5-.5h5a.5.5 0 0 1 0 1l-.224.447a1 1 0 0 1-.894.553H6.618a1 1 0 0 1-.894-.553L5.5 15a.5.5 0 0 1-.5-.5z"/></svg>
          <span class="lbl">Hint</span>
        </button>
        <button id="btn-show-solution" class="exr-nav-btn" disabled style="display:none; border-color:#6c757d; color:#6c757d;">
          <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M8 1a2 2 0 0 1 2 2v4H6V3a2 2 0 0 1 2-2zm3 6V3a3 3 0 0 0-6 0v4a2 2 0 0 0-2 2v5a2 2 0 0 0 2 2h6a2 2 0 0 0 2-2V9a2 2 0 0 0-2-2z"/></svg>
          Solution (<span id="solution-timer">30</span>s)
        </button>
      </div>

      <div id="hint-container" style="display:none;"></div>
      <div id="solution-container" style="display:none;"></div>
    </div>

    <!-- Difficulty rating: moved into the top-right corner of the question box when a question is shown -->
    <div id="difficulty-holder" hidden>
      <div id="difficulty-widget" class="exr-difficulty" title="How difficult did you find this question?">
        <span class="exr-difficulty-label">Difficulty</span>
        <div id="star-rating" class="exr-stars" role="group" aria-label="Difficulty rating">
          <button type="button" data-star="1" aria-label="Difficulty 1 of 5">☆</button><button type="button" data-star="2" aria-label="Difficulty 2 of 5">☆</button><button type="button" data-star="3" aria-label="Difficulty 3 of 5">☆</button><button type="button" data-star="4" aria-label="Difficulty 4 of 5">☆</button><button type="button" data-star="5" aria-label="Difficulty 5 of 5">☆</button>
        </div>
      </div>
    </div>

    <!-- Navigate by lecture/quiz, with the Completed / Needs review buttons in the middle -->
    <div id="section-nav" class="d-flex justify-content-between align-items-center gap-2 mt-3 flex-wrap">
          <button id="prev-sec-btn" class="exr-nav-btn prev" title="First question of the previous lecture/quiz">
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M8.354 1.646a.5.5 0 0 1 0 .708L2.707 8l5.647 5.646a.5.5 0 0 1-.708.708l-6-6a.5.5 0 0 1 0-.708l6-6a.5.5 0 0 1 .708 0z"/><path d="M12.354 1.646a.5.5 0 0 1 0 .708L6.707 8l5.647 5.646a.5.5 0 0 1-.708.708l-6-6a.5.5 0 0 1 0-.708l6-6a.5.5 0 0 1 .708 0z"/></svg>
            <span id="prev-sec-label">Previous lecture</span>
          </button>
          <div id="feedback-bar" class="exr-status" style="display:none;">
            <button id="btn-thumbs-up" type="button" class="exr-nav-btn status-up" aria-pressed="false" title="Mark as completed (click again to undo)">✓ Completed</button>
            <button id="btn-thumbs-down" type="button" class="exr-nav-btn status-down" aria-pressed="false" title="Mark as needing review (click again to undo)">⚠️ Needs review</button>
          </div>
          <button id="next-sec-btn" class="exr-nav-btn next" title="First question of the next lecture/quiz">
            <span id="next-sec-label">Next lecture</span>
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M3.646 1.646a.5.5 0 0 1 .708 0l6 6a.5.5 0 0 1 0 .708l-6 6a.5.5 0 0 1-.708-.708L9.293 8 3.646 2.354a.5.5 0 0 1 0-.708z"/><path d="M7.646 1.646a.5.5 0 0 1 .708 0l6 6a.5.5 0 0 1 0 .708l-6 6a.5.5 0 0 1-.708-.708L13.293 8 7.646 2.354a.5.5 0 0 1 0-.708z"/></svg>
          </button>
    </div>
  `;

  // DOM Elements
  const prevBtn = document.getElementById('prev-exr-btn');
  const nextBtn = document.getElementById('next-exr-btn');
  const randomBtn = document.getElementById('random-exr-btn');
  const prevSecBtn = document.getElementById('prev-sec-btn');
  const nextSecBtn = document.getElementById('next-sec-btn');
  const prevSecLabel = document.getElementById('prev-sec-label');
  const nextSecLabel = document.getElementById('next-sec-label');
  const progressEl = document.getElementById('exr-progress');
  const progressLabel = document.getElementById('exr-progress-label');
  const progressBar = document.getElementById('exr-progress-bar');
  const filterSelect = document.getElementById('filter-select');
  const sectionSelect = document.getElementById('section-select');
  const idPanel = document.getElementById('id-panel');
  const idCurrent = document.getElementById('id-current');
  const idCopyBtn = document.getElementById('id-copy');
  const idInput = document.getElementById('id-input');
  const idLoadBtn = document.getElementById('id-load');
  const idMsg = document.getElementById('id-msg');
  const displayContainer = document.getElementById('random-exercise-display');
  const feedbackBar = document.getElementById('feedback-bar');
  const btnUp = document.getElementById('btn-thumbs-up');
  const btnDown = document.getElementById('btn-thumbs-down');
  const starContainer = document.getElementById('star-rating');
  const difficultyWidget = document.getElementById('difficulty-widget');
  const manageIdBtn = document.getElementById('manage-id-btn');
  const syncStatus = document.getElementById('sync-status');

  const solutionControls = document.getElementById('solution-controls');
  const boxActions = document.getElementById('box-actions');
  const btnShowHint = document.getElementById('btn-show-hint');
  const btnShowSolution = document.getElementById('btn-show-solution');
  const solutionTimerSpan = document.getElementById('solution-timer');
  const hintContainer = document.getElementById('hint-container');
  const solutionContainer = document.getElementById('solution-container');

  const cntAll = document.getElementById('cnt-all');
  const cntUnrated = document.getElementById('cnt-unrated');
  const cntUp = document.getElementById('cnt-up');
  const cntDown = document.getElementById('cnt-down');

  let activeExerciseId = null;
  let activeLectureTitle = null;
  let showingTitleCard = false;
  let questionStartTime = null;
  let solutionTimerInterval = null;

  function getVote(id) { return localStorage.getItem('vote_' + id); }
  function getDifficulty(id) { return localStorage.getItem('diff_' + id); }
  function getTimeSpent(id) { return localStorage.getItem('timeSpent_' + id); }
  function getFirstSeen(id) { return localStorage.getItem('firstSeen_' + id); }
  function recordFirstSeen(id) {
    if (!getFirstSeen(id)) {
      localStorage.setItem('firstSeen_' + id, Date.now());
      updateStreakDisplay();
    }
  }
  function formatTimeRemaining(ms) {
    const d = Math.floor(ms / 86400000);
    const h = Math.floor((ms % 86400000) / 3600000);
    const m = Math.floor((ms % 3600000) / 60000);
    if (d > 0) return `${d}d ${h}h`;
    if (h > 0) return `${h}h ${m}m`;
    return `${m}m`;
  }

  // Release time of the solutions for an exercise (null if its lecture has no date)
  function solutionReleaseTime(exerciseElement) {
    if (exerciseElement.dataset.release === 'now') return new Date(0);
    const m = (exerciseElement.dataset.lecture || '').match(/Lecture\s+(\d+)/i);
    const lectureDate = m ? LECTURE_DATES[parseInt(m[1], 10)] : null;
    if (!lectureDate) return null;
    const d = new Date(lectureDate + 'T00:00:00Z');
    d.setUTCDate(d.getUTCDate() + SOLUTION_DELAY_DAYS);
    const releaseDate = d.toISOString().slice(0, 10);
    const offset = releaseDate >= '2026-11-01' ? '-06:00' : '-05:00'; // CST : CDT
    return new Date(`${releaseDate}T09:00:00${offset}`);
  }

  function computeStreak() {
    // Collect all unique days on which any question was first seen
    const daySet = new Set();
    allExercises.forEach(el => {
      const ts = getFirstSeen(el.id);
      if (ts) {
        const d = new Date(parseInt(ts));
        daySet.add(`${d.getFullYear()}-${d.getMonth()}-${d.getDate()}`);
      }
    });
    if (daySet.size === 0) return 0;

    // Sort days and count consecutive streak ending today (or yesterday)
    const today = new Date();
    const toKey = d => `${d.getFullYear()}-${d.getMonth()}-${d.getDate()}`;
    let streak = 0;
    let check = new Date(today);
    // Allow streak to still show if the student hasn't opened anything today yet
    if (!daySet.has(toKey(check))) check.setDate(check.getDate() - 1);
    while (daySet.has(toKey(check))) {
      streak++;
      check.setDate(check.getDate() - 1);
    }
    return streak;
  }

  function updateStreakDisplay() {
    const streak = computeStreak();
    const el = document.getElementById('streak-display');
    if (!el) return;
    if (streak === 0) { el.textContent = ''; return; }
    const flame = streak >= 3 ? '🔥' : '📅';
    el.textContent = `${flame} ${streak}d`;
    el.title = streak === 1 ? '1 day streak' : `${streak} day streak`;
  }

  async function syncProgressToServer(actionMetaData = null) {
    syncStatus.textContent = 'Syncing...';
    syncStatus.className = 'badge bg-warning text-dark border';

    const votes = {};
    const difficulties = {};
    const titles = {};
    const timeSpent = {};
    const firstSeen = {};

    allExercises.forEach(el => {
      const v = getVote(el.id);
      const d = getDifficulty(el.id);
      const t = getTimeSpent(el.id);
      const f = getFirstSeen(el.id);
      if (v) votes[el.id] = v;
      if (d) difficulties[el.id] = d;
      if (t) timeSpent[el.id] = parseInt(t, 10);
      if (f) firstSeen[el.id] = parseInt(f, 10);
      titles[el.id] = el.dataset.title;
    });

    const payload = {
      studentID: localStorage.getItem('studentID'),
      timestamp: new Date().toISOString(),
      votes: votes,
      difficulties: difficulties,
      timeSpentSeconds: timeSpent,
      firstSeenTimestamps: firstSeen,
      exerciseTitles: titles
    };

    if (actionMetaData) payload.lastAction = actionMetaData;

    try {
      const response = await fetch(SERVER_SYNC_URL, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(payload)
      });

      if (response.ok) {
        syncStatus.textContent = 'Synced';
        syncStatus.className = 'badge bg-light text-success border';
      }
    } catch (err) {
      syncStatus.textContent = 'Offline';
      syncStatus.className = 'badge bg-light text-secondary border';
    }
  }

  function updateCounters() {
    const total = allExercises.length;
    const upCount = allExercises.filter(el => getVote(el.id) === 'up').length;
    const downCount = allExercises.filter(el => getVote(el.id) === 'down').length;

    cntAll.textContent = total;
    cntUnrated.textContent = total - (upCount + downCount);
    cntUp.textContent = upCount;
    cntDown.textContent = downCount;
  }

  // Track current position as an index into allExercises (avoids findIndex bugs)
  let currentExerciseIndex = -1;

  function getActivePool() {
    const filter = filterSelect.value;
    if (filter === 'unrated') return allExercises.filter(el => !getVote(el.id));
    if (filter === 'down') return allExercises.filter(el => getVote(el.id) === 'down');
    if (filter === 'up') return allExercises.filter(el => getVote(el.id) === 'up');
    return allExercises;
  }

  function updateFeedbackUI() {
    if (!activeExerciseId || showingTitleCard) return;

    const vote = getVote(activeExerciseId);
    btnUp.classList.toggle('active', vote === 'up');
    btnDown.classList.toggle('active', vote === 'down');
    btnUp.setAttribute('aria-pressed', vote === 'up' ? 'true' : 'false');
    btnDown.setAttribute('aria-pressed', vote === 'down' ? 'true' : 'false');

    const currentDiff = parseInt(getDifficulty(activeExerciseId)) || 0;
    Array.from(starContainer.children).forEach(star => {
      const starVal = parseInt(star.dataset.star);
      star.textContent = starVal <= currentDiff ? '★' : '☆';
    });
  }

  function setupSolutionControls(exerciseElement) {
    clearInterval(solutionTimerInterval);
    
    hintContainer.style.display = 'block';
    solutionContainer.style.display = 'none';
    hintContainer.innerHTML = '';
    solutionContainer.innerHTML = '';
    btnShowHint.classList.remove('hint-on');
    btnShowHint.querySelector('.lbl').textContent = 'Hint';
    btnShowSolution.classList.remove('next', 'active');

    const hintEl = exerciseElement.querySelector('.exercise-hint');
    const solutionEl = exerciseElement.querySelector('.exercise-solution');

    if (!hintEl && !solutionEl) {
      solutionControls.style.display = 'none';
      boxActions.hidden = true;
      return;
    }
    boxActions.hidden = false;

    solutionControls.style.display = 'block';

    // Configure Hint Button — content rendered into hintContainer below the buttons
    if (hintEl) {
      btnShowHint.style.display = 'inline-flex';
      hintContainer.innerHTML = hintEl.outerHTML;
      hintContainer.querySelector('.exercise-hint').style.display = 'none';
    } else {
      btnShowHint.style.display = 'none';
    }

    // Configure Solution Button (unlocks one week after the lecture)
    if (solutionEl) {
      btnShowSolution.style.display = 'inline-flex';
      solutionContainer.innerHTML = solutionEl.innerHTML;

      const release = solutionReleaseTime(exerciseElement);
      if (!release) console.warn('No lecture date found for ' + exerciseElement.id + ' (' + exerciseElement.dataset.lecture + ')');
      const remaining = release ? release - Date.now() : Infinity;

      const lockSVG = '<svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M8 1a2 2 0 0 1 2 2v4H6V3a2 2 0 0 1 2-2zm3 6V3a3 3 0 0 0-6 0v4a2 2 0 0 0-2 2v5a2 2 0 0 0 2 2h6a2 2 0 0 0 2-2V9a2 2 0 0 0-2-2z"/></svg>';
      const unlockSVG = '<svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M11 1a2 2 0 0 0-2 2v4a2 2 0 0 1 2 2v5a2 2 0 0 1-2 2H3a2 2 0 0 1-2-2V9a2 2 0 0 1 2-2h5V3a3 3 0 0 1 6 0v4a.5.5 0 0 1-1 0V3a2 2 0 0 0-2-2z"/></svg>';

      if (remaining <= 0) {
        // Already unlocked
        btnShowSolution.disabled = false;
        btnShowSolution.className = 'exr-nav-btn next';
        btnShowSolution.style.cssText = '';
        btnShowSolution.innerHTML = unlockSVG + ' <span class="lbl">Show solution</span>';
      } else {
        // Locked — show the release date and time remaining, no live countdown needed
        btnShowSolution.disabled = true;
        btnShowSolution.className = 'exr-nav-btn';
        btnShowSolution.style.cssText = 'border-color:#6c757d; color:#6c757d; opacity:0.6; cursor:not-allowed;';
        if (release) {
          const releaseStr = release.toLocaleDateString(undefined, { weekday: 'short', day: 'numeric', month: 'short' });
          btnShowSolution.innerHTML = lockSVG + ' Solution available ' + releaseStr + ' (in ' + formatTimeRemaining(remaining) + ')';
        } else {
          btnShowSolution.innerHTML = lockSVG + ' Solution not yet available';
        }
      }
    } else {
      btnShowSolution.style.display = 'none';
    }
  }

  // Questions are copied from the (hidden) question list, which MathJax typesets when the page
  // loads, so their maths is usually already rendered. Only typeset content that still contains
  // raw TeX, once MathJax has finished starting up.
  function typeset(el, attempt = 0) {
    if (el.querySelector('mjx-container') || !/\\\(|\\\[/.test(el.textContent)) return;
    const mj = window.MathJax;
    if (mj && mj.typesetPromise) {
      mj.typesetPromise([el]).catch(err => console.warn('MathJax:', err));
    } else if (attempt < 50) {
      setTimeout(() => typeset(el, attempt + 1), 200);   // try again for up to 10 seconds
    }
  }

  // 1. A question shown before MathJax has typeset the (hidden) question list is copied with raw
  // TeX. Wait until MathJax has typeset its source, then show it again (once), copied from the
  // typeset source. Polls every 200 ms for up to 20 s.
  function refreshWhenTypeset(src, attempt = 0) {
    if (activeExerciseId !== src.id) return;                                  // moved on
    if (displayContainer.querySelector('mjx-container') ||
        !/\\\(|\\\[/.test(displayContainer.textContent)) return;               // no raw TeX shown
    if (src.querySelector('mjx-container')) {                                  // source is typeset now
      renderExercise(src, { history: 'none', refreshed: true });
      return;
    }
    if (attempt < 100) setTimeout(() => refreshWhenTypeset(src, attempt + 1), 200);
  }

  function renderExercise(chosenElement, opts = {}) {
    if (!chosenElement) {
      displayContainer.innerHTML = `<div class="alert alert-info m-0">No exercises found for this filter.</div>`;
      feedbackBar.style.display = 'none';
      solutionControls.style.display = 'none';
      activeExerciseId = null;
      showingTitleCard = false;
      questionStartTime = null;
      updateSectionNav();
      return;
    }

    showingTitleCard = false;
    feedbackBar.style.display = 'flex';
    activeExerciseId = chosenElement.id;
    activeLectureTitle = chosenElement.dataset.lecture;
    currentExerciseIndex = allExercises.indexOf(chosenElement);
    questionStartTime = Date.now();
    recordFirstSeen(chosenElement.id);

    const clone = chosenElement.cloneNode(true);
    clone.style.display = 'block';

    // Keep only .exercise-solution stripped from clone; hint is shown below via button
    clone.querySelectorAll('.exercise-hint, .exercise-solution').forEach(e => e.remove());

    clone.prepend(difficultyWidget);      // difficulty stars in the top-right corner of the question box
    clone.appendChild(boxActions);        // Hint / Solution buttons at the bottom of the question box

    displayContainer.innerHTML = '';
    displayContainer.appendChild(clone);

    // Setup hint and solution triggers (must be after clone is in DOM)
    setupSolutionControls(chosenElement);

    if (!opts.refreshed) refreshWhenTypeset(chosenElement);

    updateFeedbackUI();
    updateSectionNav();

    // 2. remember this question, and give it its own address (so Back/Forward and links work)
    try { localStorage.setItem('bank_lastExercise', chosenElement.id); } catch (e) {}
    const url = '#' + chosenElement.id;
    if (opts.history !== 'none' && location.hash !== url) {
      if (opts.history === 'replace') history.replaceState(null, '', url);
      else history.pushState(null, '', url);
    }
  }

  // Hint Click Handler — toggles .exercise-hint rendered below the buttons
  btnShowHint.addEventListener('click', () => {
    const inlineHint = hintContainer.querySelector('.exercise-hint');
    if (!inlineHint) return;
    const isVisible = inlineHint.style.display !== 'none';
    if (isVisible) {
      inlineHint.style.display = 'none';
      btnShowHint.classList.remove('hint-on');
      btnShowHint.querySelector('.lbl').textContent = 'Hint';
    } else {
      inlineHint.style.display = '';
      btnShowHint.classList.add('hint-on');
      btnShowHint.querySelector('.lbl').textContent = 'Hide hint';
      typeset(inlineHint);
      syncProgressToServer({ exerciseID: activeExerciseId, requestedHint: true });
    }
  });

  // Solution Click Handler — toggles solution visibility and active pill state
  btnShowSolution.addEventListener('click', () => {
    const isVisible = solutionContainer.style.display === 'block';
    const lbl = btnShowSolution.querySelector('.lbl');
    if (isVisible) {
      solutionContainer.style.display = 'none';
      btnShowSolution.classList.remove('active');
      if (lbl) lbl.textContent = 'Show solution';
    } else {
      solutionContainer.style.display = 'block';
      btnShowSolution.classList.add('active');
      if (lbl) lbl.textContent = 'Hide solution';
      if (window.hljs) hljs.highlightAll();
      typeset(solutionContainer);
      syncProgressToServer({ exerciseID: activeExerciseId, requestedSolution: true });
    }
  });

  function showNextExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    // Find where current exercise sits in the pool by index in allExercises
    const posInPool = pool.findIndex(el => allExercises.indexOf(el) === currentExerciseIndex);
    const nextPos = posInPool === -1 ? 0 : (posInPool + 1) % pool.length;
    renderExercise(pool[nextPos]);
  }

  function showPrevExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    const posInPool = pool.findIndex(el => allExercises.indexOf(el) === currentExerciseIndex);
    const prevPos = posInPool === -1 ? pool.length - 1 : (posInPool - 1 + pool.length) % pool.length;
    renderExercise(pool[prevPos]);
  }

  // ── Navigation by lecture/quiz ("section") ──────────────────────────────────
  // Sections in the order they appear in the bank, e.g. Lecture 1, ..., Lecture 4, Quiz 1, Lecture 5
  const sections = [...new Set(allExercises.map(el => el.dataset.lecture))];

  function currentSectionIndex() {
    if (currentExerciseIndex < 0) return -1;
    return sections.indexOf(allExercises[currentExerciseIndex].dataset.lecture);
  }

  // Nearest section in direction dir (+1 or -1) with at least one question in the active pool
  function neighbourSection(dir) {
    const pool = getActivePool();
    const cur = currentSectionIndex();
    for (let k = cur + dir; k >= 0 && k < sections.length; k += dir) {
      const first = pool.find(el => el.dataset.lecture === sections[k]);
      if (first) return { name: sections[k], first: first };
    }
    return null;
  }

  function showNextSection() {
    const target = neighbourSection(+1);
    if (target) renderExercise(target.first);
  }

  function showPrevSection() {
    const target = neighbourSection(-1);
    if (target) renderExercise(target.first);
  }

  sections.forEach(name => {
    const opt = document.createElement('option');
    opt.value = name; opt.textContent = name;
    sectionSelect.appendChild(opt);
  });
  sectionSelect.addEventListener('change', () => {
    const name = sectionSelect.value;
    let target = getActivePool().find(el => el.dataset.lecture === name);
    if (!target) {                        // nothing in this section matches the filter: show all questions
      filterSelect.value = 'all';
      updateCounterButtons();
      target = allExercises.find(el => el.dataset.lecture === name);
    }
    if (target) renderExercise(target);
  });

  function updateSectionNav() {
    if (currentExerciseIndex >= 0) sectionSelect.value = allExercises[currentExerciseIndex].dataset.lecture;
    const prev = neighbourSection(-1), next = neighbourSection(+1);
    prevSecBtn.disabled = !prev;
    nextSecBtn.disabled = !next;
    prevSecLabel.textContent = prev ? prev.name : 'Previous lecture';
    nextSecLabel.textContent = next ? next.name : 'Next lecture';

    updateProgressBar();
  }

  // One segment per question of the current lecture/quiz: green = completed, amber = needs
  // review, light blue = unrated questions before the current one; the current question is
  // outlined. Click a segment to jump to that question.
  function updateProgressBar() {
    if (currentExerciseIndex < 0 || !activeExerciseId) { progressEl.hidden = true; return; }
    const el = allExercises[currentExerciseIndex];
    const inSection = allExercises.filter(e => e.dataset.lecture === el.dataset.lecture);
    const pos = inSection.indexOf(el);
    const pool = getActivePool();

    progressEl.hidden = false;
    progressLabel.textContent = el.dataset.lecture;
    progressBar.setAttribute('aria-label', `${el.dataset.lecture}: question ${pos + 1} of ${inSection.length}`);
    progressBar.setAttribute('aria-valuenow', pos + 1);
    progressBar.setAttribute('aria-valuemax', inSection.length);
    progressBar.innerHTML = '';
    inSection.forEach((ex, k) => {
      const seg = document.createElement('button');
      seg.type = 'button';
      const vote = getVote(ex.id);
      seg.className = 'exr-seg' + (vote === 'up' ? ' up' : vote === 'down' ? ' down' : k < pos ? ' passed' : '') + (k === pos ? ' current' : '');
      const status = getVote(ex.id) === 'up' ? ' (completed)' : getVote(ex.id) === 'down' ? ' (needs review)' : '';
      seg.title = `Question ${k + 1} of ${inSection.length}: ${ex.dataset.title}${status}`;
      seg.setAttribute('aria-label', seg.title);
      if (!pool.includes(ex)) seg.disabled = true;          // hidden by the current filter
      else if (k !== pos) seg.addEventListener('click', () => renderExercise(ex));
      progressBar.appendChild(seg);
    });
  }

  function showRandomExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    const chosen = pool.length > 1 ? pool.filter(el => el.id !== activeExerciseId)[Math.floor(Math.random() * (pool.length - 1))] : pool[0];
    renderExercise(chosen);
  }

  function handleStatusClick(newVoteState) {
    if (!activeExerciseId || showingTitleCard) return;

    let timeSpentSecs = null;
    if (questionStartTime) {
      const elapsedSeconds = Math.round((Date.now() - questionStartTime) / 1000);
      if (elapsedSeconds > 10) {
        timeSpentSecs = elapsedSeconds;
        const existingTime = parseInt(getTimeSpent(activeExerciseId) || '0', 10);
        localStorage.setItem('timeSpent_' + activeExerciseId, existingTime + timeSpentSecs);
      }
    }

    const currentVote = getVote(activeExerciseId);
    currentVote === newVoteState 
      ? localStorage.removeItem('vote_' + activeExerciseId) 
      : localStorage.setItem('vote_' + activeExerciseId, newVoteState);

    updateFeedbackUI();
    updateCounters();
    updateProgressBar();
    syncProgressToServer({ exerciseID: activeExerciseId, statusMarked: newVoteState, timeSpentOnThisSession: timeSpentSecs });
  }

  btnUp.addEventListener('click', () => handleStatusClick('up'));
  btnDown.addEventListener('click', () => handleStatusClick('down'));

  starContainer.addEventListener('click', (e) => {
    if (!activeExerciseId || showingTitleCard) return;
    const starVal = e.target.closest('[data-star]') && e.target.closest('[data-star]').dataset.star;
    if (!starVal) return;
    getDifficulty(activeExerciseId) === starVal ? localStorage.removeItem('diff_' + activeExerciseId) : localStorage.setItem('diff_' + activeExerciseId, starVal);
    updateFeedbackUI();
    syncProgressToServer();
  });

  const counterButtons = Array.from(document.querySelectorAll('.cnt-btn'));
  function updateCounterButtons() {
    counterButtons.forEach(b => b.classList.toggle('active', b.dataset.filter === filterSelect.value && filterSelect.value !== 'all'));
  }
  counterButtons.forEach(b => b.addEventListener('click', () => {
    filterSelect.value = b.dataset.filter;
    filterSelect.dispatchEvent(new Event('change'));
  }));

  filterSelect.addEventListener('change', () => {
    updateCounterButtons();
    // When filter changes, reset to first exercise in the new pool
    // (not 'next from current' which breaks when current isn't in new pool)
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    // If the current exercise is in the new pool, keep it; otherwise go to first
    const stillInPool = pool.find(el => el.id === activeExerciseId);
    renderExercise(stillInPool || pool[0]);
  });
  prevBtn.addEventListener('click', showPrevExercise);
  prevSecBtn.addEventListener('click', showPrevSection);
  nextSecBtn.addEventListener('click', showNextSection);
  nextBtn.addEventListener('click', showNextExercise);
  randomBtn.addEventListener('click', showRandomExercise);

  async function fetchAndRestoreProgress(id) {
    syncStatus.textContent = 'Syncing...';
    syncStatus.className = 'badge bg-warning text-dark border';
    try {
      const response = await fetch(`${SERVER_SYNC_URL}?studentID=${encodeURIComponent(id)}`);
      if (!response.ok) throw new Error('Not found');
      const data = await response.json();
      if (data.status === 'error') throw new Error(data.message);

      // Merge server data into localStorage — server wins on conflicts
      let restored = 0;
      Object.entries(data.votes || {}).forEach(([k, v]) => { localStorage.setItem('vote_' + k, v); restored++; });
      Object.entries(data.difficulties || {}).forEach(([k, v]) => { localStorage.setItem('diff_' + k, v); });
      Object.entries(data.timeSpentSeconds || {}).forEach(([k, v]) => { localStorage.setItem('timeSpent_' + k, v); });
      Object.entries(data.firstSeenTimestamps || {}).forEach(([k, v]) => { localStorage.setItem('firstSeen_' + k, v); });

      // Update all UI that reads from localStorage
      updateCounters();
      updateStreakDisplay();
      updateFeedbackUI();
      // Re-render current exercise so the UI reflects the restored progress
      if (currentExerciseIndex >= 0) renderExercise(allExercises[currentExerciseIndex]);

      syncStatus.textContent = 'Synced';
      syncStatus.className = 'badge bg-success text-white border';
      showIdMessage(`Progress loaded (${restored} rated question${restored === 1 ? '' : 's'}).`, 'success');
      return true;
    } catch (e) {
      syncStatus.textContent = 'Offline';
      syncStatus.className = 'badge bg-danger text-white border';
      showIdMessage('Could not load progress for this ID: ' + e.message, 'danger');
      return false;
    }
  }

  function showIdMessage(text, kind) {
    idMsg.textContent = text;
    idMsg.className = 'mt-2 text-' + (kind || 'muted');
  }

  manageIdBtn.addEventListener('click', () => {
    const open = idPanel.hidden;
    idPanel.hidden = !open;
    manageIdBtn.setAttribute('aria-expanded', open ? 'true' : 'false');
    if (open) {
      idCurrent.textContent = localStorage.getItem('studentID') || '(none)';
      showIdMessage('', 'muted');
      idInput.value = '';
    }
  });

  idCopyBtn.addEventListener('click', () => {
    const id = idCurrent.textContent;
    const done = () => showIdMessage('Copied ' + id + ' to the clipboard.', 'success');
    if (navigator.clipboard && navigator.clipboard.writeText) {
      navigator.clipboard.writeText(id).then(done, () => showIdMessage('Copy failed: select the ID and copy it by hand.', 'danger'));
    } else {
      showIdMessage('Select the ID and copy it by hand.', 'muted');
    }
  });

  function loadFromId() {
    const trimmed = idInput.value.trim();
    if (trimmed === '') return;
    if (!/^\d{3}-\d{3}-\d{3}$/.test(trimmed)) {
      showIdMessage('That doesn\'t look like an ID: it should look like 042-317-891.', 'danger');
      return;
    }
    localStorage.setItem('studentID', trimmed);
    studentID = trimmed;
    idCurrent.textContent = trimmed;
    showIdMessage('Loading…', 'muted');
    // Fetch existing progress for this ID from server, then push local data up
    fetchAndRestoreProgress(trimmed).then(() => syncProgressToServer({ changedID: true }));
  }
  idLoadBtn.addEventListener('click', loadFromId);
  idInput.addEventListener('keydown', e => { if (e.key === 'Enter') loadFromId(); });

  // ── 8. keyboard shortcuts ───────────────────────────────────────────────────
  document.addEventListener('keydown', e => {
    if (e.ctrlKey || e.metaKey || e.altKey) return;
    const t = e.target;
    if (t && (t.isContentEditable || /^(INPUT|SELECT|TEXTAREA)$/.test(t.tagName))) return;   // typing in a box
    if (e.key === 'ArrowRight') { e.preventDefault(); e.shiftKey ? showNextSection() : showNextExercise(); }
    else if (e.key === 'ArrowLeft') { e.preventDefault(); e.shiftKey ? showPrevSection() : showPrevExercise(); }
    else if (e.key === 's' || e.key === 'S') { showRandomExercise(); }
    else if ((e.key === 'h' || e.key === 'H') && btnShowHint.style.display !== 'none') { btnShowHint.click(); }
  });

  // Back / Forward buttons (and edited addresses) show the matching question
  window.addEventListener('popstate', () => {
    const el = allExercises.find(ex => '#' + ex.id === location.hash);
    if (el && el.id !== activeExerciseId) renderExercise(el, { history: 'none' });
  });

  updateCounters();
  updateCounterButtons();
  updateStreakDisplay();

  // Start with the question in the address (links from the lectures use bank.html#bank-exr-...),
  // otherwise the last question viewed in this browser, otherwise a random one.
  const hashId = window.location.hash.slice(1); // e.g. 'bank-exr-pi-artan'
  let lastId = null;
  try { lastId = localStorage.getItem('bank_lastExercise'); } catch (e) {}
  const startEl = (hashId && allExercises.find(el => el.id === hashId)) ||
                  (lastId && allExercises.find(el => el.id === lastId));
  if (startEl) {
    renderExercise(startEl, { history: 'replace' });
  } else {
    const pool = getActivePool();
    renderExercise(pool[Math.floor(Math.random() * pool.length)] || null, { history: 'replace' });
  }


});
</script>